The script will take in the bronze tables to create the master tables for each path, 

In [0]:
from pyspark.sql.functions import col, to_date

data_timeline = "202607"                # to use in the table names, adjust as needed
# create the silver schema beforehand
silver_schema_path = "2026_july_balance.silver_schema"

# gather other schemas
bronze_schema_path = "`2026_july_balance`.bronze_schema"
code_desc_schema_path = "workspace.code_desc_project_master"

all_bronze_tables = spark.sql(f"SHOW TABLES IN {bronze_schema_path}").toPandas()["tableName"].tolist()
fund_names = sorted(set(name.split("_")[2] for name in all_bronze_tables))
print(fund_names)



In [0]:
print(all_bronze_tables)

In [0]:
for fund in fund_names:
    print(f"processing {fund}")

    try:
        
        balance = spark.table(f"{bronze_schema_path}.balances_{data_timeline}_{fund}")
        balance = balance.withColumn("Effective_Date", to_date(col("Effective_Date"), "yyyyMMdd"))
        balance = balance.withColumn("Amount", col("Amount").cast("double"))
        
        print(f"{fund} balance table has {balance.count()} records")
        
        # Start joining tables together...
        master = balance            #set balance the master table

        # Join accounts table
        try:
            accounts = spark.table(f"{bronze_schema_path}.accounts_{data_timeline}_{fund}")
            print(f"{fund} accounts table has {accounts.count()} records")
            accounts_cols = [col for col in accounts.columns if col not in balance.columns or col in ['Account_Code', 'Entity_Code']]
            master = master.join(accounts.select(*accounts_cols), on=['Account_Code', 'Entity_Code'], how="left")
            print(f"Done joining accounts table for {fund}")

        except:
            print(f"{fund} accounts table not found")


        # Join departments table - parent department descriptions are available so pull them in
        try:
            departments = spark.table(f"{bronze_schema_path}.departments_{data_timeline}_{fund}")
            print(f"{fund} departments table has {departments.count()} records")
            
            # Get the parent department descriptions
            dept_parent = departments.select(col("Entity_Code"), col("Department_Code"), col("Department_Description").alias("Parent_Dept_Description")) 
            departments_w_parent = departments.join(dept_parent, on =
                                            (departments.Entity_Code == dept_parent.Entity_Code) &
                                            (departments.Parent_Department_Code  == dept_parent.Department_Code),
                                            how="left").drop(dept_parent.Entity_Code, dept_parent.Department_Code)
            
            # Now join with master
            departments_cols = [col for col in departments_w_parent.columns if col not in balance.columns or col in ['Department_Code', 'Entity_Code']]
            master = master.join(departments_w_parent.select(*departments_cols), on=['Department_Code', 'Entity_Code'], how="left")
            print(f"Done joining departments table for {fund}")

        except:
            print(f"{fund} departments table not found")

        
        # Join funds table. - parent fund codes are not of the same type.
        try:
            funds_data = spark.table(f"{bronze_schema_path}.funds_{data_timeline}_{fund}")
            print(f"{fund} funds table has {funds_data.count()} records")
            funds_cols = [col for col in funds_data.columns if col not in balance.columns or col in ['Fund_Code', 'Entity_Code']]
            master = master.join(funds_data.select(*funds_cols), on=['Fund_Code', 'Entity_Code'], how="left")
            print(f"Done joining funds table for {fund}")

        except:
            print(f"{fund} funds table not found")
        
        # Join projects table
        try:
            projects = spark.table(f"{bronze_schema_path}.projects_{data_timeline}_{fund}")
            print(f"{fund} projects table has {projects.count()} records")
            projects = projects.withColumn("Project_Start_Date", to_date(col("Project_Start_Date"), "yyyyMMdd")).withColumn     ("Project_End_Date", to_date(col("Project_End_Date"), "yyyyMMdd"))

            project_cols = [col for col in projects.columns if col not in balance.columns or col in ['Project_Code', 'Entity_Code']]

            master = master.join(projects.select(*project_cols), on=['Project_Code', 'Entity_Code'], how="left")
            print(f"Done joining projects table for {fund}")

        except:
            print(f"{fund} projects table not found")


        # Save master table
        # master_table_name = f"workspace.silver_fund_data.{fund}_master"
        # master.write.format("delta") \
        #       .mode("overwrite") \
        #       .option("overwriteSchema", "true") \
        #       .saveAsTable(master_table_name)

        # print(f"Created {master_table_name} table")
        print(f"{fund} silver table created")
        display(master)

    except Exception as e:
        print(f"Failed to access {fund}: {e}")


print("Silver layer created and uploaded")
  